# BJJ Detection Model Training

Upload `training_data_detection_all_cameras.zip` to Google Drive under `roll_tracker_training/` before running.

For Kaggle: change `DRIVE_PATH` to `/kaggle/input/datasets/bryanrt/roll-tracker-training/` and skip Cell 1.

In [ ]:
# Cell 1 — Setup (Colab)
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

In [ ]:
# Cell 2 — Unpack training data
DRIVE_PATH = "/content/drive/MyDrive/roll_tracker_training"
TRAINING_ZIP = f"{DRIVE_PATH}/training_data_detection_all_cameras.zip"

!unzip -q {TRAINING_ZIP} -d /content/training_data/

from pathlib import Path
target = Path("/content/training_data")
print(f"Training data at: {target}")
print(f"Images: {len(list(target.glob('images/*')))}")
print(f"Labels: {len(list(target.glob('labels/*')))}")

In [ ]:
# Cell 3 — Download stock detection model (auto-download via ultralytics)
from ultralytics import YOLO

model = YOLO("yolo26n.pt")  # Downloads automatically if not present
print(f"Model loaded: {model.model_name}")

In [ ]:
# Cell 4 — Fix dataset paths for Colab/Kaggle
import yaml
from pathlib import Path

dataset_yaml = Path("/content/training_data/dataset.yaml")
config = yaml.safe_load(dataset_yaml.read_text())

config["path"] = "/content/training_data"
dataset_yaml.write_text(yaml.dump(config, default_flow_style=False))

# Rewrite train.txt and val.txt with absolute paths
for split_file in ["train.txt", "val.txt"]:
    split_path = Path(f"/content/training_data/{split_file}")
    lines = split_path.read_text().strip().split("\n")
    new_lines = []
    for line in lines:
        filename = Path(line).name
        new_lines.append(f"/content/training_data/images/{filename}")
    split_path.write_text("\n".join(new_lines) + "\n")

print("Dataset paths updated for Colab")
print(f"Config: {yaml.dump(config)}")

In [ ]:
# Cell 5 — Train
from ultralytics import YOLO

# === CONFIGURE ===
BASE_MODEL = "yolo26n.pt"
FREEZE = 10
EPOCHS = 100
LR0 = 0.001
ROUND_NAME = "detection_all_cameras"
# =================

model = YOLO(BASE_MODEL)

results = model.train(
    data="/content/training_data/dataset.yaml",
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    device=0,
    freeze=FREEZE,
    lr0=LR0,
    project=f"/content/training_runs/{ROUND_NAME}",
    name="train",
    exist_ok=True,
    save=True,
    plots=True,
)

rd = getattr(results, "results_dict", {})
print(f"\n{'='*50}")
print(f"TRAINING COMPLETE \u2014 {ROUND_NAME}")
print(f"{'='*50}")
print(f"Box mAP50:     {rd.get('metrics/mAP50(B)', 0):.4f}")
print(f"Box mAP50-95:  {rd.get('metrics/mAP50-95(B)', 0):.4f}")

In [ ]:
# Cell 6 — Save model to Drive
import shutil
from pathlib import Path

best_pt = Path(f"/content/training_runs/{ROUND_NAME}/train/weights/best.pt")
if not best_pt.exists():
    best_pt = list(Path(f"/content/training_runs/{ROUND_NAME}").rglob("best.pt"))[0]

output_name = "bjj-detect-all-cameras.pt"
output_path = f"{DRIVE_PATH}/{output_name}"
shutil.copy2(best_pt, output_path)
print(f"Model saved to Drive: {output_path}")

# Save training plots
plots_dir = best_pt.parent.parent
for plot_file in plots_dir.glob("*.png"):
    shutil.copy2(plot_file, f"{DRIVE_PATH}/{ROUND_NAME}_{plot_file.name}")
    print(f"Saved plot: {plot_file.name}")